**RoBERTa-base Fine-Tune Script**

This is script was created to fine-tune a 100M parameter specialized model. This model will be fine-tuned with a large dataset of 284k phishing and safe emails

## Model

In [ ]:
!pip install transformers torch pandas scikit-learn datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import pandas as pd
import numpy as np
import os
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

# Paths
TRAIN_DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/phishing_project/datasets/cleaned/fine_tuning/train_combined.csv'
TEST_DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/phishing_project/datasets/cleaned/test/clean_combined.csv'
MODEL_SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/phishing_project/models/roberta_finetuned/'
RESULTS_PATH = '/content/drive/MyDrive/Colab Notebooks/phishing_project/results/'

# Create directories if they don't exist
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

MODEL_NAME = "FacebookAI/roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATE = 2e-5


In [ ]:
#Model

In [ ]:
print("Loading training data...")
df = pd.read_csv(TRAIN_DATA_PATH)
print(f"Training dataset: {df.shape}")
print(df['label'].value_counts())

# Split into train and validation
train_data, val_data = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df['label']
)
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print(f"\nTrain: {len(train_data)} emails")
print(f"Validation: {len(val_data)} emails")
print(f"\nTrain label distribution:")
print(train_data['label'].value_counts())

# Compute class weights to handle imbalance
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_data['label']
)
print(f"\nClass weights: {class_weights}")

Loading training data...
Training dataset: (284264, 2)
label
0    199766
1     84498
Name: count, dtype: int64

Train: 255837 emails
Validation: 28427 emails

Train label distribution:
label
0    179789
1     76048
Name: count, dtype: int64

Class weights: [0.71149236 1.68207579]


Dataset Class

In [ ]:
class EmailDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df['email_text'][idx])
        label = int(self.df['label'][idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

print("Dataset class defined")

Dataset class defined


Load Model - Create Dataloaders

In [ ]:
print("Loading RoBERTa tokenizer and model...")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
model = model.to("cuda")
print("Model loaded")

# Create datasets
train_dataset = EmailDataset(train_data, tokenizer, MAX_LENGTH)
val_dataset = EmailDataset(val_data, tokenizer, MAX_LENGTH)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

# Class weights for loss function
weights = torch.tensor(class_weights, dtype=torch.float).to("cuda")
loss_fn = torch.nn.CrossEntropyLoss(weight=weights)

print(f"Total training steps: {total_steps}")
print("Ready to fine-tune")

Loading RoBERTa tokenizer and model...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded
Train batches: 7995
Val batches: 889
Total training steps: 23985
Ready to fine-tune


Fine-tune loop

In [ ]:
print("Starting fine-tuning...\n")

best_val_f1 = 0
best_model_path = f"{MODEL_SAVE_PATH}best_model/"
os.makedirs(best_model_path, exist_ok=True)

for epoch in range(EPOCHS):
    # ---- Training ----
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to("cuda")
        attention_mask = batch['attention_mask'].to("cuda")
        labels = batch['label'].to("cuda")

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # Print progress every 200 batches
        if (batch_idx + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | "
                  f"Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {total_loss/(batch_idx+1):.4f} | "
                  f"Acc: {correct/total*100:.2f}%")

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total * 100

    # ---- Validation ----
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to("cuda")
            attention_mask = batch['attention_mask'].to("cuda")
            labels = batch['label'].to("cuda")

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds) * 100
    val_f1 = f1_score(val_labels, val_preds)
    val_loss = val_loss / len(val_loader)

    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS} Complete")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"Val F1:     {val_f1:.4f}")
    print(f"{'='*60}\n")

    # Save best model based on validation F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"New best model saved with F1: {val_f1:.4f}")

# Save final model
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)
print(f"\nFinal model saved to {MODEL_SAVE_PATH}")
print(f"Best validation F1: {best_val_f1:.4f}")

# Clear memory
del model
torch.cuda.empty_cache()
print("Model cleared from memory")

Starting fine-tuning...

Epoch 1/3 | Batch 200/7995 | Loss: 0.6669 | Acc: 70.44%
Epoch 1/3 | Batch 400/7995 | Loss: 0.4366 | Acc: 82.34%
Epoch 1/3 | Batch 600/7995 | Loss: 0.3245 | Acc: 87.20%
Epoch 1/3 | Batch 800/7995 | Loss: 0.2660 | Acc: 89.73%
Epoch 1/3 | Batch 1000/7995 | Loss: 0.2328 | Acc: 91.29%
Epoch 1/3 | Batch 1200/7995 | Loss: 0.2062 | Acc: 92.38%
Epoch 1/3 | Batch 1400/7995 | Loss: 0.1864 | Acc: 93.25%
Epoch 1/3 | Batch 1600/7995 | Loss: 0.1729 | Acc: 93.88%
Epoch 1/3 | Batch 1800/7995 | Loss: 0.1612 | Acc: 94.38%
Epoch 1/3 | Batch 2000/7995 | Loss: 0.1506 | Acc: 94.82%
Epoch 1/3 | Batch 2200/7995 | Loss: 0.1415 | Acc: 95.19%
Epoch 1/3 | Batch 2400/7995 | Loss: 0.1345 | Acc: 95.47%
Epoch 1/3 | Batch 2600/7995 | Loss: 0.1283 | Acc: 95.72%
Epoch 1/3 | Batch 2800/7995 | Loss: 0.1230 | Acc: 95.93%
Epoch 1/3 | Batch 3000/7995 | Loss: 0.1172 | Acc: 96.15%
Epoch 1/3 | Batch 3200/7995 | Loss: 0.1127 | Acc: 96.33%
Epoch 1/3 | Batch 3400/7995 | Loss: 0.1082 | Acc: 96.50%
Epoch 1/3 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New best model saved with F1: 0.9937
Epoch 2/3 | Batch 200/7995 | Loss: 0.0173 | Acc: 99.72%
Epoch 2/3 | Batch 400/7995 | Loss: 0.0121 | Acc: 99.77%
Epoch 2/3 | Batch 600/7995 | Loss: 0.0120 | Acc: 99.79%
Epoch 2/3 | Batch 800/7995 | Loss: 0.0122 | Acc: 99.78%
Epoch 2/3 | Batch 1000/7995 | Loss: 0.0117 | Acc: 99.79%
Epoch 2/3 | Batch 1200/7995 | Loss: 0.0113 | Acc: 99.79%
Epoch 2/3 | Batch 1400/7995 | Loss: 0.0116 | Acc: 99.78%
Epoch 2/3 | Batch 1600/7995 | Loss: 0.0117 | Acc: 99.78%
Epoch 2/3 | Batch 1800/7995 | Loss: 0.0117 | Acc: 99.78%
Epoch 2/3 | Batch 2000/7995 | Loss: 0.0120 | Acc: 99.77%
Epoch 2/3 | Batch 2200/7995 | Loss: 0.0122 | Acc: 99.76%
Epoch 2/3 | Batch 2400/7995 | Loss: 0.0124 | Acc: 99.76%
Epoch 2/3 | Batch 2600/7995 | Loss: 0.0124 | Acc: 99.76%
Epoch 2/3 | Batch 2800/7995 | Loss: 0.0124 | Acc: 99.76%
Epoch 2/3 | Batch 3000/7995 | Loss: 0.0128 | Acc: 99.75%
Epoch 2/3 | Batch 3200/7995 | Loss: 0.0132 | Acc: 99.74%
Epoch 2/3 | Batch 3400/7995 | Loss: 0.0131 | Acc: 99.74

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New best model saved with F1: 0.9963
Epoch 3/3 | Batch 200/7995 | Loss: 0.0027 | Acc: 99.92%
Epoch 3/3 | Batch 400/7995 | Loss: 0.0041 | Acc: 99.91%
Epoch 3/3 | Batch 600/7995 | Loss: 0.0037 | Acc: 99.92%
Epoch 3/3 | Batch 800/7995 | Loss: 0.0032 | Acc: 99.93%
Epoch 3/3 | Batch 1000/7995 | Loss: 0.0028 | Acc: 99.94%
Epoch 3/3 | Batch 1200/7995 | Loss: 0.0027 | Acc: 99.94%
Epoch 3/3 | Batch 1400/7995 | Loss: 0.0023 | Acc: 99.95%
Epoch 3/3 | Batch 1600/7995 | Loss: 0.0021 | Acc: 99.95%
Epoch 3/3 | Batch 1800/7995 | Loss: 0.0025 | Acc: 99.95%
Epoch 3/3 | Batch 2000/7995 | Loss: 0.0025 | Acc: 99.94%
Epoch 3/3 | Batch 2200/7995 | Loss: 0.0026 | Acc: 99.94%
Epoch 3/3 | Batch 2400/7995 | Loss: 0.0029 | Acc: 99.94%
Epoch 3/3 | Batch 2600/7995 | Loss: 0.0026 | Acc: 99.95%
Epoch 3/3 | Batch 2800/7995 | Loss: 0.0025 | Acc: 99.95%
Epoch 3/3 | Batch 3000/7995 | Loss: 0.0025 | Acc: 99.95%
Epoch 3/3 | Batch 3200/7995 | Loss: 0.0024 | Acc: 99.95%
Epoch 3/3 | Batch 3400/7995 | Loss: 0.0025 | Acc: 99.95

Upload to Hugging Face

In [ ]:
from huggingface_hub import login, HfApi
from google.colab import userdata

token = userdata.get('WHF_TOKEN')
login(token=token)

api = HfApi()
api.upload_folder(
    folder_path='/content/drive/MyDrive/Colab Notebooks/phishing_project/models/roberta_finetuned/best_model/',
    repo_id="eduardocastellon/RoBERTa-base",
    repo_type="model",
    commit_message="Upload fine-tuned RoBERTa phishing detector",
    token=token # Explicitly pass the token
)

print("Upload complete!")
print("Model available at: https://huggingface.co/eduardocastellon/RoBERTa-base")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t_model/model.safetensors:   0%|          |  552kB /  499MB            

Upload complete!
Model available at: https://huggingface.co/eduardocastellon/RoBERTa-base


In [ ]:
print("Checking token and repository information...")
try:
    # Get information about the currently logged-in user
    user_info = api.whoami()
    print(f"Logged in as: {user_info['name']} (User ID: {user_info['id']})")

    # List repositories owned by the logged-in user
    print("\nListing your model repositories...")
    repos = api.list_models(author=user_info['name'])
    found_repo = False
    for repo in repos:
        print(f"  - {repo.id}")
        if repo.id == "eduardocastellon/RoBERTa-base": # Replace with your target repo_id if different
            found_repo = True

    if not found_repo:
        print("\nWarning: The target repository 'eduardocastellon/RoBERTa-base' was not found in your list of models.")
        print("Please ensure the repository name is correct and it's created under your account on Hugging Face Hub, or that your token has permission to create it.")
    else:
        print("\nTarget repository 'eduardocastellon/RoBERTa-base' found.")

except Exception as e:
    print(f"An error occurred while checking repositories: {e}")
    print("Please ensure your token is valid and has sufficient permissions (read access at minimum for listing). Make sure it's stored in Colab Secrets as 'WHF_TOKEN'.")

Checking token and repository information...
Logged in as: eduardocastellon (User ID: 69e5812caae91aa7506c53ca)

Listing your model repositories...
  - eduardocastellon/RoBERTa-base

Target repository 'eduardocastellon/RoBERTa-base' found.
